# Run simple BAO fits, using the BAO parameterization

In this notebook we run MCMC chains for BAO likelihoods, both DESI-Y5 forecasts as well as unrealistic fake BAO data

In [1]:
import numpy as np
import sys
from pathlib import Path

In [2]:
# figure out configuration for laptop vs NERSC
laptop=True
if laptop:
    cosmo_dir = '/Users/afont/Desktop/bao-cosmology'
else:
    cosmo_dir = '/global/cfs/cdirs/desi/users/font/bao-cosmology'
sys.path.insert(1, f'{cosmo_dir}/py/')
import write_cobaya_configs as write

In [3]:
# set configuration options for these runs
cosmo_path = Path(cosmo_dir)

# forecasted DESI measurements
desi_path = cosmo_path / 'data/desi-y5'
desi_labels = ['BGS', 'LRG', 'ELG', 'QSO', 'LYA']
print(desi_path)

# mock BAO measurements
mock_path = cosmo_path / 'data/mock_bao'
mock_labels = ['mock_z0.001_bao', 'mock_z10.0_ap', 'mock_cmb_bao']
print(mock_path)

# store chains here
runs_path = cosmo_path / 'runs'

/Users/afont/Desktop/bao-cosmology/data/desi-y5
/Users/afont/Desktop/bao-cosmology/data/mock_bao


### Use full DESI-Y5 forecast to constraint cosmological models

In [ ]:
if laptop:
    # if running at NERSC, run analyses for all cosmological models
    model='lcdm'
    config_file = write.write_multiple_bao_config(model=model, run_label='desi-y5',
                                                  bao_labels=desi_labels, data_path=desi_path,
                                                  runs_path=runs_path)
    print(config_file)
    updated_info, sampler = write.run_cobaya(config_file, resume=False, force=True)
else:
    # if running at NERSC, run analyses for all cosmological models
    for model in ['lcdm', 'olcdm', 'nulcdm', 'w0wa']:
        config_file = write.write_multiple_bao_config(model=model, run_label='desi-y5',
                                                      bao_labels=desi_labels, data_path=desi_path,
                                                      runs_path=runs_path)
        command = write.send_cobaya_job(config_file, debug=True, submit=True)

wrote cobaya config file /Users/afont/Desktop/bao-cosmology/runs/bao/lcdm/desi-y5/cobaya_config.yaml
/Users/afont/Desktop/bao-cosmology/runs/bao/lcdm/desi-y5/cobaya_config.yaml
[output] Output to be read-from/written-into folder '/Users/afont/Desktop/bao-cosmology/runs/bao/lcdm/desi-y5', with prefix 'chain'
[output] Found existing info files with the requested output prefix: '/Users/afont/Desktop/bao-cosmology/runs/bao/lcdm/desi-y5/chain'
[output] Will delete previous products ('force' was requested).
[camb] `camb` module loaded successfully from /Users/afont/Codes/cobaya-packages/code/CAMB/camb
[bgs] Initialized.
[lrg] Initialized.
[elg] Initialized.
[qso] Initialized.
[lya] Initialized.
[mcmc] Getting initial point... (this may take a few seconds)
[mcmc] Initial point: hrdrag:98.99869, omm:0.3134124
[model] Measuring speeds... (this may take a few seconds)
[model] Setting measured speeds (per sec): {BGS: 6430.0, LRG: 4880.0, ELG: 6860.0, QSO: 6910.0, LYA: 2540.0, camb.transfers: 257.

### Use DESI-Y5 forecast one target at a time (BGS, LRG, ELG, QSO, LYA)

In [ ]:
for label in desi_labels:
    config_file = write.write_single_bao_config(model='lcdm', label=label,
                                            data_path=desi_path, runs_path=runs_path)
    if laptop:
        updated_info, sampler = write.run_cobaya(config_file, resume=True)
    else:
        command = write.send_cobaya_job(config_file, debug=True, submit=True)
        print(command)

### Use toy BAO measurements (for pedagogical reasons)

In [ ]:
for label in mock_labels:
    config_file = write.write_single_bao_config(model='lcdm', label=label,
                                            data_path=mock_path, runs_path=runs_path)
    if laptop:
        updated_info, sampler = write.run_cobaya(config_file, resume=True)
    else:
        command = write.send_cobaya_job(config_file, debug=True, submit=True)
        print(command)